# Data Understanding and Preparation 

Covers **Stage 1 (Dataset Understanding + EDA)** and **Stage 2 (Data Preparation)**. 
- Each stage below lists its **sub-tasks as a checklist**
- Complete every `# TODO` so each checklist item is satisfied
- Implement the reusable logic in `src/data_prep.py` and call it here so the same code backs your notebook and the later stages

**File ownership** — 
- *Provided (no change needed):* `config.py` (paths, target, column lists, params)
- *Provided to extend:* `src/data_prep.py` (function stubs)
- *You create:* the cells below + the report sections

**Business context:** flag diabetic inpatients at risk of 30-day readmission so care teams can intervene. The target is highly imbalanced (~9%), so metric choice and leakage prevention matter.

# Stage 1 - Business & Data Understanding, EDA, and Methodology  <font color="red">[20 marks]</font>

> **Stage 1 also has report-only tasks** graded from the MLOps report, not this notebook: **1.1 Business Understanding** (1.1.1–1.1.3) and **1.4 Methodology Design** (1.4.1–1.4.2).

In [ ]:
%pip install pandas
%pip install matplotlib
%pip install seaborn
%pip install sklearn

In [1]:
%load_ext autoreload
%autoreload 2

In [1]:
import warnings; warnings.filterwarnings('ignore')
import pandas as pd, numpy as np, matplotlib.pyplot as plt, seaborn as sns
import src.data_prep as dp
import config as cfg

# TODO: load data/diabetic_data.csv treating '?' as NaN (na_values=['?']); print shape; head(3).
raw = dp.load_raw()
print(raw.shape)
raw.head(3)


(101766, 50)


,encounter_id,patient_nbr,race,gender,age,weight,admission_type_id,discharge_disposition_id,admission_source_id,time_in_hospital,...,citoglipton,insulin,glyburide-metformin,glipizide-metformin,glimepiride-pioglitazone,metformin-rosiglitazone,metformin-pioglitazone,change,diabetesMed,readmitted
0,2278392,8222157,Caucasian,Female,[0-10),NaN,6,25,1,1,...,No,No,No,No,No,No,No,No,No,NO
1,149190,55629189,Caucasian,Female,[10-20),NaN,1,1,7,3,...,No,Up,No,No,No,No,No,Ch,Yes,>30
2,64410,86047875,AfricanAmerican,Female,[20-30),NaN,1,1,7,2,...,No,No,No,No,No,No,No,No,Yes,NO


## Stage 1.2 — Dataset Understanding <font color="red">[4 marks]</font>

- **1.2.1 — Dataset structure, target derivation & patients-vs-encounters [2]** — print the shape (~101,766 × 50), derive the target (`readmitted` `<30` → 1), and compare unique patients (`patient_nbr`) vs encounters.
- **1.2.2 — Feature types & data-quality / missing-value quantification [2]** — summarise feature types and build a per-column missing-value table with percentages (note `?` was loaded as NaN; e.g. `weight` ~97% missing).

In [4]:
# TODO 1.2.1: print rows/columns; compare unique patients vs encounters; derive the binary target.
# TODO 1.2.2: print feature dtypes summary and a missing-value table (count + pct) per column.
print(f"Shape: {raw.shape[0]:,} rows x {raw.shape[1]} columns")

# Compare unique patients vs encounters
num_encounters = len(raw)
num_patients = raw['patient_nbr'].nunique()
print(f"Encounters: {num_encounters:,}")
print(f"Unique patients: {num_patients:,}")
print(f"Encounters per patient (avg): {num_encounters / num_patients:.2f}")

# Readmitted patients within 30 days
clean = dp.clean(raw)
print("\nAfter clean() - first-encounter + hospice/expired removal:")
print(f"Cleaned Shape: {clean.shape[0]:,} rows x {clean.shape[1]} columns")
print(f"Rows removed:  {len(raw) - len(clean):,} " 
      f"({(1 - len(clean) / len(raw)):.1%} of raw encounters)")

print(f"\nTarget rate (from clean()): "
      f"{clean[cfg.TARGET].mean():.2%} positive "
      f"({clean[cfg.TARGET].sum():,} of {len(clean):,})")

# missing-value table (count + pct) per column
missing_raw = pd.DataFrame({
    "dtype": raw.dtypes.astype(str),
    "missing_count": raw.isna().sum(),
    "missing_pct": (raw.isna().mean() * 100).round(2),
}).sort_values("missing_pct", ascending=False)

# Feature types
print("\nDtype counts (raw):")
print(raw.dtypes.value_counts())

print("\nColumns with missing values:")
missing_raw[missing_raw["missing_count"] > 0]

# What clean() removes
dropped_by_clean = raw.columns.difference(clean.columns)
print(f"Columns removed by clean(): {sorted(dropped_by_clean)}")


Shape: 101,766 rows x 50 columns
Encounters: 101,766
Unique patients: 71,518
Encounters per patient (avg): 1.42

After clean() - first-encounter + hospice/expired removal:
Cleaned Shape: 69,990 rows x 38 columns
Rows removed:  31,776 (31.2% of raw encounters)

Target rate (from clean()): 8.98% positive (6,285 of 69,990)

Dtype counts (raw):
object    37
int64     13
Name: count, dtype: int64

Columns with missing values:
Columns removed by clean(): ['acetohexamide', 'citoglipton', 'encounter_id', 'examide', 'glimepiride-pioglitazone', 'metformin-pioglitazone', 'metformin-rosiglitazone', 'patient_nbr', 'payer_code', 'readmitted', 'tolbutamide', 'troglitazone', 'weight']


## Stage 1.3 — Exploratory Data Analysis <font color="red">[7 marks]</font>

- **1.3.1 — Univariate analysis [3]** — visualise the target imbalance and ≥3 numeric feature distributions, with brief observations.
- **1.3.2 — Bivariate analysis vs target [2]** — analyse ≥1 relationship to the target (e.g. readmission rate vs prior inpatient visits) and interpret it.
- **1.3.3 — Visualisations + written interpretation [2]** — clear, labelled plots; interpret each key finding in writing (see the interpretation cell below).

In [5]:
# TODO 1.3.1: plot the target imbalance + >=3 numeric distributions.
# TODO 1.3.2: plot >=1 bivariate view vs the target (e.g. readmit rate by prior inpatient visits).
# TODO 1.3.3: ensure plots are labelled; record findings in the interpretation cell.


### ✍️ Interpretation (1.3.3)
*TODO: what does the ~9% imbalance imply for metric choice? Which columns are unusable (e.g. weight ~97% missing) and why? What does the bivariate view tell you?*

## Stage 2 — Data Preparation  <font color="red">[25 marks]</font>

## Stage 2.1 — Data Cleaning <font color="red">[6 marks]</font>

Implement `clean()` in `src/data_prep.py`, then call it here.

- **2.1.1 — Missing value handling & invalid value treatment [2]** — `?`→NaN; drop `weight` and constant/zero-variance medication columns.
- **2.1.2 — Patient deduplication & removal of invalid records [2]** — keep the FIRST encounter per patient (`drop_duplicates('patient_nbr')`, prevents leakage); drop expired/hospice discharges (`discharge_disposition_id` ∈ {11,13,14,19,20,21}).
- **2.1.3 — Target variable creation & validation [2]** — build the binary target and validate its class balance (expect ~69,973 rows, ~9% positive).

In [6]:
# TODO (src/data_prep.py clean()): implement sub-tasks 2.1.1–2.1.3 above.
# TODO (here): clean = dp.clean(raw); print the resulting shape and target rate.


## Stage 2.2 — Feature Engineering <font color="red">[8 marks]</font>

Implement `engineer_features()` in `src/data_prep.py`.

- **2.2.1 — ICD-9 diagnosis grouping into clinical categories [4]** — group `diag_1/2/3` into clinical categories (circulatory, respiratory, diabetes, injury, other) with a justified mapping.
- **2.2.2 — Additional engineered features, justified [4]** — ≥3 of {age midpoint, service_utilization, num_med_changes, medical_specialty top-k grouping}, each justified.

In [7]:
# TODO (src/data_prep.py engineer_features()): implement sub-tasks 2.2.1–2.2.2 above.
# TODO (here): fe = dp.engineer_features(clean); preview the new feature columns.


## Stage 2.3 — Feature Transformation <font color="red">[5 marks]</font>

Implement `build_preprocessor()` in `src/data_prep.py`.

- **2.3.1 — Numeric scaling & categorical encoding [3]** — numeric: median-impute + scale; categorical: impute + `OneHotEncoder(handle_unknown='ignore')`.
- **2.3.2 — Leakage-safe ColumnTransformer/Pipeline assembly [2]** — assemble ONE ColumnTransformer; do **not** fit it yet (fitting happens on the train split only, in 2.4).

In [8]:
# TODO (src/data_prep.py build_preprocessor()): implement sub-tasks 2.3.1–2.3.2 above.
# TODO (here): construct the (unfitted) preprocessor and show its structure.


## Stage 2.4 — Validation & Splitting <font color="red">[6 marks]</font>

Implement `get_splits()` in `src/data_prep.py`.

- **2.4.1 — Stratified train/validation/test split created first [3]** — STRATIFIED split BEFORE any fitting (`random_state=42`; class ratio preserved).
- **2.4.2 — Leakage-safe fit-on-train + class-balance check [3]** — fit the preprocessor on TRAIN only, transform val/test; confirm the class ratio is preserved across splits.

In [9]:
# TODO (src/data_prep.py get_splits()): implement sub-tasks 2.4.1–2.4.2 above.
# TODO (here): create splits; fit preprocessor on train only; print split sizes,
#       feature count, and the class ratio per split (confirm no leakage).
